# 15 — Storage Fill Rate Statistics
Monthly and yearly EU gas storage statistics, fullness breakdown, and withdrawal sign validation.

**Run notebook 01 first** to generate `data/processed/eu_aggregate_full.parquet`.

In [ ]:
import sys, os, warnings
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from pathlib import Path
warnings.filterwarnings('ignore')

for _c in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (_c / 'src' / 'agsi_client').exists():
        sys.path.insert(0, str(_c))
        os.chdir(_c)
        print(f"Project root: {_c}")
        break

# ═══════════════════════════════════════════════════════
# CONFIGURATION  — change YEAR here
YEAR = 2024     # int → single year overlay; None → all years
# ═══════════════════════════════════════════════════════

DATA = Path("data/processed")
df = pd.read_parquet(DATA / "eu_aggregate_full.parquet")
df.index = pd.to_datetime(df.index).tz_localize(None)
df = df.sort_index()
df["year"]  = df.index.year
df["month"] = df.index.month
df["doy"]   = df.index.day_of_year
# ── Season year: winter spans Oct of year N through Mar of year N+1 ──────────
# Summer N = Apr 1 N  →  Sep 30 N     (month 4-9,  season_year = N)
# Winter N = Oct 1 N  →  Mar 31 N+1   (month 10-12 → season_year=N;
#                                       month 1-3  → season_year = calendar_year - 1)
df["season_year"] = df["year"].copy()
df.loc[df["month"] <= 3, "season_year"] -= 1
df["season"] = df["month"].map(lambda m: "Summer" if 4 <= m <= 9 else "Winter")

YEARS = sorted(df["year"].unique())
print(f"Loaded: {df.shape}  |  {df.index[0].date()} -> {df.index[-1].date()}")
print(f"Years:  {YEARS}")
print(f"YEAR selector: {YEAR if YEAR else 'All (overlay mode)'}")
print(f"Columns: {list(df.columns)}")


## 0 · CRITICAL: Withdrawal Sign & Direction Validation

Verifies that `withdrawal` is a **positive** GWh/day value representing gas **removed** from storage, and that `gasInStorage` moves in the correct direction each season.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CRITICAL VALIDATION — withdrawal sign / direction
# ══════════════════════════════════════════════════════════════════════════════
#
# AGSI convention (confirmed in client.py parsing):
#   injection   GWh/day — POSITIVE = gas added to storage     (Apr-Sep)
#   withdrawal  GWh/day — POSITIVE = gas removed from storage (Oct-Mar)
#   gasInStorage TWh    — RISES during Summer, FALLS during Winter
#
# Season year conventions used throughout this notebook:
#   Summer N : Apr 1 N  →  Sep 30 N    (season_year = N, months 4-9)
#   Winter N : Oct 1 N  →  Mar 31 N+1  (season_year = N, months 10-12 of N
#                                                          months 1-3 of N+1)
#
# Net balance: Δ gasInStorage (TWh) ≈ (injection − withdrawal) / 1000
# ══════════════════════════════════════════════════════════════════════════════

# ── 1a. Basic positivity check ────────────────────────────────────────────────
winter_mask = df["season"] == "Winter"
summer_mask = df["season"] == "Summer"

wd_winter  = df.loc[winter_mask, "withdrawal"].dropna()
inj_summer = df.loc[summer_mask, "injection"].dropna()

pct_wd_pos  = (wd_winter > 0).mean() * 100
pct_inj_pos = (inj_summer > 0).mean() * 100

print("=" * 70)
print("  WITHDRAWAL / INJECTION SIGN VALIDATION")
print("=" * 70)
print(f"\n  withdrawal column (Winter = Oct-Mar):")
print(f"    % days withdrawal > 0 : {pct_wd_pos:.1f}%   (expect ~80-95%)")
print(f"    Mean on active days   : {wd_winter[wd_winter>0].mean():,.0f} GWh/day")
print(f"    Min value             : {wd_winter.min():.1f}  (negative = sign error!)")
print(f"    Max value             : {wd_winter.max():.1f}")

print(f"\n  injection column (Summer = Apr-Sep):")
print(f"    % days injection > 0  : {pct_inj_pos:.1f}%   (expect ~85-95%)")
print(f"    Mean on active days   : {inj_summer[inj_summer>0].mean():,.0f} GWh/day")
print(f"    Min value             : {inj_summer.min():.1f}")

# ── 1b. Direction check using season_year ──────────────────────────────────
print(f"\n{'─' * 70}")
print("  gasInStorage direction per season (season_year boundaries):")
recent_sys = sorted(df["season_year"].unique())[-7:]
for sy in recent_sys:
    for season, expected_sign in [("Summer", +1), ("Winter", -1)]:
        mask = (df["season_year"] == sy) & (df["season"] == season)
        sub = df[mask].sort_index()
        stor = sub["gasInStorage"].dropna()
        if len(stor) < 5:
            continue
        delta = stor.iloc[-1] - stor.iloc[0]
        ok = "✅" if delta * expected_sign > 0 else "❌ WRONG SIGN"
        period = (f"Apr-Sep {sy}" if season == "Summer"
                  else f"Oct {sy}-Mar {sy+1}")
        print(f"    {season:<6} {sy}  [{period}]  ΔStorage: {delta:+7.1f} TWh  {ok}")

# ── 1c. Season-year energy balance ────────────────────────────────────────────
print(f"\n{'─' * 70}")
print("  Energy balance by season (season_year corrects Winter year boundary)")
print(f"  {'SY':<5} {'Season':<7} {'Period':<22} "
      f"{'Inj(TWh)':>10} {'Wd(TWh)':>9} {'Net':>8} {'ΔActual':>9} {'Gap':>7} {'OK?':>5}")
print(f"  {'─' * 78}")

rows_ok = rows_gap = 0
for sy in sorted(df["season_year"].unique()):
    for season in ("Summer", "Winter"):
        mask = (df["season_year"] == sy) & (df["season"] == season)
        s = df[mask].sort_index()
        if len(s) < 20:
            continue
        total_inj = s["injection"].clip(lower=0).sum() / 1000
        total_wd  = s["withdrawal"].clip(lower=0).sum() / 1000
        net_calc  = total_inj - total_wd
        stor      = s["gasInStorage"].dropna()
        delta     = float(stor.iloc[-1] - stor.iloc[0]) if len(stor) >= 2 else float("nan")
        gap       = net_calc - delta
        ok = "✅" if abs(gap) < 15 else ("⚠️" if abs(gap) < 40 else "❌")
        period = (f"Apr-Sep {sy}" if season == "Summer"
                  else f"Oct {sy}-Mar {sy+1}")
        print(f"  {sy:<5} {season:<7} {period:<22} "
              f"{total_inj:>10.1f} {total_wd:>9.1f} {net_calc:>8.1f} "
              f"{delta:>9.1f} {gap:>7.1f} {ok:>5}")
        if ok == "✅":
            rows_ok += 1
        else:
            rows_gap += 1

print(f"\n  Summary: {rows_ok} seasons ✅   {rows_gap} seasons ⚠️/❌")
print(f"\n  CONCLUSION:")
if rows_gap == 0:
    print("  ✅ Both columns are POSITIVE GWh/day; seasons balance correctly.")
elif pct_wd_pos < 50:
    print("  ❌ withdrawal appears to be negative — sign error in source data.")
else:
    print("  ⚠️  Small gaps found — normal (linepack, timing, missing country data).")
    print("     No sign error detected. Gaps < 40 TWh are within expected range.")


## 1 · Monthly Fill Rate Statistics (all years)
Mean, median, std, min, max, P10, P25, P75, P90 fill % by calendar month.

In [ ]:
MONTH_NAMES = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

# ── Compute monthly statistics across ALL years ───────────────────────────────
monthly = df.groupby("month")["full"].agg(
    Mean="mean", Median="median", Std="std",
    Min="min", Max="max",
    P10=lambda x: x.quantile(0.10),
    P25=lambda x: x.quantile(0.25),
    P75=lambda x: x.quantile(0.75),
    P90=lambda x: x.quantile(0.90),
    N="count",
).round(1)
monthly.index = MONTH_NAMES

print("Monthly EU Storage Fill % Statistics (all years)")
print("=" * 75)
print(monthly.to_string())

# ── Bar chart: mean with P25-P75 error bars ───────────────────────────────────
fig = go.Figure()
fig.add_trace(go.Bar(
    x=monthly.index,
    y=monthly["Mean"],
    error_y=dict(
        type="data",
        symmetric=False,
        array=(monthly["P75"] - monthly["Mean"]).values,
        arrayminus=(monthly["Mean"] - monthly["P25"]).values,
        visible=True,
        color="#555",
    ),
    marker_color="#2E75B6",
    name="Mean fill %",
    hovertemplate=(
        "<b>%{x}</b><br>"
        "Mean: %{y:.1f}%<br>"
        "P25–P75: see error bars<extra></extra>"
    ),
))
# P10-P90 as a secondary range (markers)
fig.add_trace(go.Scatter(
    x=monthly.index, y=monthly["P10"],
    mode="markers", name="P10",
    marker=dict(symbol="triangle-down", size=8, color="#E74C3C"),
))
fig.add_trace(go.Scatter(
    x=monthly.index, y=monthly["P90"],
    mode="markers", name="P90",
    marker=dict(symbol="triangle-up", size=8, color="#27AE60"),
))
fig.add_hline(y=90, line_dash="dot", line_color="purple", line_width=1,
              annotation_text="90% Target")
fig.update_layout(
    title="EU Gas Storage Fill % — Monthly Statistics (all years)<br>"
          "<sup>Bar = mean | Error bars = P25–P75 | Triangles = P10 / P90</sup>",
    xaxis_title="Month", yaxis_title="Fill (%)",
    yaxis=dict(range=[0, 105]),
    height=440, template="plotly_white",
    legend=dict(orientation="h", y=-0.15),
)
fig.show()


## 2 · Year Selector
Set `YEAR` in the setup cell.
- **Integer** (e.g. `2024`): overlay that year's monthly fill vs all-time average
- **None**: overlay all years on day-of-year axis

In [ ]:
# ── YEAR = int → overlay selected year against all-time monthly mean ──────────
# ── YEAR = None → overlay all years on day-of-year axis ──────────────────────

import plotly.colors as pc
COLORS = pc.qualitative.Set1 + pc.qualitative.Set2 + pc.qualitative.Plotly

if YEAR is not None:
    # Single year vs historical average
    yr_df = df[df["year"] == YEAR]
    if yr_df.empty:
        print(f"No data for year {YEAR}. Available: {YEARS}")
    else:
        # Monthly means for selected year
        yr_monthly = yr_df.groupby("month")["full"].mean().reindex(range(1, 13))
        yr_monthly.index = MONTH_NAMES

        fig = go.Figure()
        # All-time mean + P25-P75 band
        fig.add_trace(go.Scatter(
            x=list(range(12)) + list(range(11, -1, -1)),
            y=list(monthly["P75"]) + list(monthly["P25"][::-1]),
            fill="toself", fillcolor="rgba(46,117,182,0.15)",
            line=dict(color="rgba(0,0,0,0)"), name="P25–P75 range", showlegend=True,
        ))
        fig.add_trace(go.Scatter(
            x=MONTH_NAMES, y=monthly["Mean"],
            name="All-time average",
            line=dict(color="#2E75B6", width=2, dash="dash"),
        ))
        fig.add_trace(go.Scatter(
            x=MONTH_NAMES, y=yr_monthly,
            name=str(YEAR),
            line=dict(color="#E74C3C", width=2.8),
            mode="lines+markers",
            marker=dict(size=7),
        ))
        fig.add_hline(y=90, line_dash="dot", line_color="purple", line_width=1,
                      annotation_text="90% Target")
        fig.update_layout(
            title=f"EU Storage Fill % — {YEAR} vs All-Time Monthly Average",
            xaxis_title="Month", yaxis_title="Fill (%)",
            yaxis=dict(range=[0, 105]),
            height=440, template="plotly_white",
            legend=dict(orientation="h", y=-0.15),
        )
        fig.show()

        # Print deviation from average
        diff = (yr_monthly - monthly["Mean"]).round(1)
        print(f"  {YEAR} monthly fill vs all-time average (pp deviation):")
        for m, d in diff.items():
            bar = "▲" if d > 0 else "▼"
            print(f"    {m:<4}: {d:+5.1f} pp {bar}")

else:
    # All years overlaid on day-of-year axis
    fig = go.Figure()
    # P10-P90 background band
    bands = df.groupby("doy")["full"].quantile([0.10, 0.90]).unstack(level=1)
    fig.add_trace(go.Scatter(
        x=list(bands.index) + list(bands.index[::-1]),
        y=list(bands[0.90]) + list(bands[0.10][::-1]),
        fill="toself", fillcolor="rgba(46,117,182,0.10)",
        line=dict(color="rgba(0,0,0,0)"), name="P10–P90 band",
    ))
    for i, yr in enumerate(YEARS):
        yr_data = df[df["year"] == yr]
        lw      = 2.5 if yr == YEARS[-1] else 1.0
        opacity = 1.0 if yr >= YEARS[-1] - 2 else 0.4
        fig.add_trace(go.Scatter(
            x=yr_data["doy"], y=yr_data["full"],
            name=str(yr), mode="lines",
            line=dict(color=COLORS[i % len(COLORS)], width=lw),
            opacity=opacity,
        ))
    tick_vals  = [1, 32, 60, 91, 121, 152, 182, 213, 244, 274, 305, 335]
    tick_text  = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
    fig.add_hline(y=90, line_dash="dot", line_color="black", line_width=1,
                  annotation_text="90% Target")
    fig.update_layout(
        title="EU Storage Fill % — All Years Overlay",
        xaxis=dict(title="Day of Year", tickvals=tick_vals, ticktext=tick_text),
        yaxis=dict(title="Fill (%)", range=[0, 105]),
        height=500, template="plotly_white",
    )
    fig.show()


## 3 · Average Fullness Statistics
All-time averages, seasonal split (injection vs withdrawal), and fill % at key dates each year.

In [ ]:
# ── All-time average fill % ───────────────────────────────────────────────────
fill = df["full"].dropna()
print("=" * 60)
print("  EU STORAGE FULLNESS — OVERALL STATISTICS")
print("=" * 60)
print(f"  Date range:      {fill.index[0].date()} -> {fill.index[-1].date()}")
print(f"  N obs:           {len(fill):,} days")
print(f"  All-time mean:   {fill.mean():.2f}%")
print(f"  All-time median: {fill.median():.2f}%")
print(f"  Std dev:         {fill.std():.2f}%")
print(f"  Min:             {fill.min():.2f}%  ({fill.idxmin().date()})")
print(f"  Max:             {fill.max():.2f}%  ({fill.idxmax().date()})")

# ── By season ────────────────────────────────────────────────────────────────
print(f"\n{'─' * 60}")
print("  Average fill % by season:")
for season, mask in [
    ("Summer (Apr-Sep)", df["season"] == "Summer"),
    ("Withdrawal season (Oct-Mar)", df["month"].isin([11, 12, 1, 2, 3])),
]:
    s = df.loc[mask, "full"].dropna()
    print(f"  {season}:")
    print(f"    Mean={s.mean():.1f}%  Median={s.median():.1f}%  "
          f"Std={s.std():.1f}%  P10={s.quantile(.10):.1f}%  P90={s.quantile(.90):.1f}%")

# ── Fill % at key dates each year ────────────────────────────────────────────
print(f"\n{'─' * 60}")
print("  Fill % at key seasonal dates (nearest available day):")
key_dates = {
    "Apr 1":  (4, 1),
    "Oct 1":  (10, 1),
    "Nov 1":  (11, 1),
}
rows = []
for yr in YEARS:
    row = {"Year": yr}
    for label, (mo, dy) in key_dates.items():
        target = pd.Timestamp(yr, mo, dy)
        window = df[(df.index >= target - pd.Timedelta(days=3)) &
                    (df.index <= target + pd.Timedelta(days=3))]["full"]
        row[label] = round(float(window.mean()), 1) if not window.empty else float("nan")
    rows.append(row)

key_tbl = pd.DataFrame(rows).set_index("Year")
print(key_tbl.to_string())

# ── Chart: key-date fill % over years ────────────────────────────────────────
key_colors = {"Apr 1": "#27AE60", "Oct 1": "#E67E22", "Nov 1": "#E74C3C"}
fig = go.Figure()
for col, color in key_colors.items():
    fig.add_trace(go.Scatter(
        x=key_tbl.index, y=key_tbl[col],
        name=col, mode="lines+markers",
        line=dict(color=color, width=2),
        marker=dict(size=6),
    ))
fig.add_hline(y=90, line_dash="dot", line_color="purple", line_width=1,
              annotation_text="90%")
fig.update_layout(
    title="EU Storage Fill % at Key Seasonal Dates (Apr 1 / Oct 1 / Nov 1) by Year",
    xaxis_title="Year", yaxis_title="Fill (%)",
    yaxis=dict(range=[0, 105]),
    height=420, template="plotly_white",
    legend=dict(orientation="h", y=-0.15),
)
fig.show()


## 4 · Seasonal Flow Totals
Year-by-year injection and withdrawal volumes (TWh), cross-checked against actual storage changes.
Confirms that `withdrawal > 0` correctly represents gas removed.

In [ ]:
# ── Seasonal flow totals using correct season_year boundaries ─────────────────
# Summer N = Apr-Sep of year N    (season_year=N, season="Summer")
# Winter N = Oct of year N + Jan-Mar of year N+1 (season_year=N, season="Winter")
#
# Both columns are POSITIVE GWh/day:
#   injection  > 0 = gas added to storage
#   withdrawal > 0 = gas removed from storage
#   Net Calc = injection - withdrawal  should ≈  ΔgasInStorage

season_rows = []
for sy in sorted(df["season_year"].unique()):
    for season in ("Summer", "Winter"):
        mask = (df["season_year"] == sy) & (df["season"] == season)
        s = df[mask].sort_index()
        if len(s) < 20:
            continue
        total_inj = s["injection"].clip(lower=0).sum() / 1000
        total_wd  = s["withdrawal"].clip(lower=0).sum() / 1000
        net_calc  = total_inj - total_wd
        stor      = s["gasInStorage"].dropna()
        delta     = float(stor.iloc[-1] - stor.iloc[0]) if len(stor) >= 2 else np.nan
        period    = (f"Apr-Sep {sy}" if season == "Summer"
                     else f"Oct {sy}-Mar {sy+1}")
        season_rows.append({
            "SY":           sy,
            "Season":       season,
            "Period":       period,
            "Inj (TWh)":   round(total_inj, 1),
            "Wd (TWh)":    round(total_wd, 1),
            "Net (TWh)":   round(net_calc, 1),
            "ΔStorage (TWh)": round(delta, 1) if not np.isnan(delta) else np.nan,
            "Gap (TWh)":   round(net_calc - delta, 1) if not np.isnan(delta) else np.nan,
        })

tbl = pd.DataFrame(season_rows)
print("EU Storage — Seasonal Flow Summary (TWh, season_year boundaries)")
print("=" * 78)
print(tbl.set_index(["SY","Season","Period"]).to_string())
print()
print("Net (TWh) ≈ ΔStorage (TWh)  is the key check.")
print("Summer: Net should be positive (storage filling).")
print("Winter: Net should be negative (storage draining).")

# ── Chart: Summer injection vs Winter withdrawal by season_year ───────────────
summer_rows = [r for r in season_rows if r["Season"] == "Summer"]
winter_rows = [r for r in season_rows if r["Season"] == "Winter"]

fig = go.Figure()
fig.add_trace(go.Bar(
    x=[r["SY"] for r in summer_rows],
    y=[r["Inj (TWh)"] for r in summer_rows],
    name="Summer Injection (Apr-Sep TWh)", marker_color="#27AE60",
))
fig.add_trace(go.Bar(
    x=[r["SY"] for r in winter_rows],
    y=[-r["Wd (TWh)"] for r in winter_rows],
    name="Winter Withdrawal (Oct-Mar TWh, shown negative)", marker_color="#E74C3C",
))
fig.add_hline(y=0, line_color="black", line_width=1)
fig.update_layout(
    title="EU Gas Storage: Summer Injection vs Winter Withdrawal by Season Year",
    barmode="group",
    xaxis_title="Season Year (Winter N = Oct N → Mar N+1)",
    yaxis_title="TWh (injection positive, withdrawal negative)",
    height=440, template="plotly_white",
    legend=dict(orientation="h", y=-0.18),
)
fig.show()
